# ELA-1 — Phishing & Malicious Email Classifier
**Track 1 | Deep Learning Applications**

This notebook implements an end-to-end pipeline that accepts raw email text and predicts **Legitimate** or **Phishing/Malicious**.

### Requirements covered
- Python 3.x + PyTorch
- **2,000 synthetic labeled emails** (requirement: at least 1,500)
- Text cleaning, tokenisation, padding/truncation
- Sequential **Bidirectional LSTM**
- Loss/accuracy curves
- Confusion matrix
- Precision, Recall and F1-score
- Raw-email inference demo

**Note:** A 5% label-noise injection makes the synthetic benchmark less trivial. Because the dataset is synthetic, the results are a demonstration of the pipeline, not production-grade phishing detection.

In [ ]:
import random, re, os
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix, classification_report
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

SEED=42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)
device=torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("PyTorch:",torch.__version__," | Device:",device)

## 1. Generate the synthetic dataset

In [ ]:
legit_subjects=["Project update","Meeting reminder","Invoice for review","Team lunch","Weekly report","Account statement","Travel confirmation","Document shared","Schedule update","Performance review"]
legit_bodies=[
"Hi {name}, please review the attached {doc} before {day}.",
"The meeting is scheduled for {time} on {day}. Let me know if you need changes.",
"Attached is the {doc} for your records. No action is required today.",
"Thanks for your help with the project. We can discuss this in our next meeting.",
"Your monthly statement is available in the company portal.",
"Please find the updated schedule attached.",
"The team will meet in the conference room at {time}."
]
phish_subjects=["Urgent account verification required","Security alert: unusual login","Your account will be suspended","Payment failed - action required","You have won a reward","Password expires today","Confirm your identity immediately","Final notice: account access"]
phish_bodies=[
"Urgent! Your account has been flagged. Verify your identity immediately at {url} to avoid suspension.",
"We detected unusual activity. Click {url} and enter your password and verification code within 24 hours.",
"Your payment failed. Update your billing information at {url} immediately or your service will be terminated.",
"Congratulations! You have been selected for a reward. Claim it at {url} before the offer expires.",
"Your password expires today. Confirm your credentials at {url} to keep access.",
"This is the final warning. Failure to click {url} will result in permanent account closure."
]
names=["Alex","Sam","Priya","Jordan","Ravi","Maya","Daniel","Sara"]
docs=["report","proposal","invoice","presentation","spreadsheet"]
days=["Monday","Tuesday","Wednesday","Friday"]
times=["9 AM","11 AM","2 PM","4 PM"]
urls=["http://secure-login.example.com","http://verify-account.example.net","http://billing-check.example.org"]

rows=[]
for i in range(2000):
    label=i%2
    if label==0:
        subject=random.choice(legit_subjects)
        body=random.choice(legit_bodies).format(name=random.choice(names),doc=random.choice(docs),day=random.choice(days),time=random.choice(times),url=random.choice(urls))
    else:
        subject=random.choice(phish_subjects)
        body=random.choice(phish_bodies).format(url=random.choice(urls))
    text=f"Subject: {subject}. {body}"
    if random.random()<0.25: text += " Please respond as soon as possible." if label else " Thank you."
    if random.random()<0.12: text=text.replace("http://","https://")
    rows.append((text,label))
random.shuffle(rows)

# 5% label noise
for j in random.sample(range(len(rows)),int(.05*len(rows))):
    rows[j]=(rows[j][0],1-rows[j][1])

df=pd.DataFrame(rows,columns=["text","label"])
print(df["label"].value_counts())
df.head()

## 2. Preprocessing: cleaning, tokenisation, padding and truncation

In [ ]:
indices=np.arange(len(df))
train_idx,test_idx=train_test_split(indices,test_size=.20,stratify=df["label"],random_state=SEED)
train_idx,val_idx=train_test_split(train_idx,test_size=.125,stratify=df.loc[train_idx,"label"],random_state=SEED)

def clean_text(text):
    return re.sub(r"[^a-z0-9]+"," ",text.lower()).strip()

counter=Counter(word for i in train_idx for word in clean_text(df.iloc[i]["text"]).split())
MAX_VOCAB=5000
MAX_LEN=60
vocab={"<PAD>":0,"<UNK>":1}
for word,_ in counter.most_common(MAX_VOCAB):
    vocab[word]=len(vocab)

def encode(text):
    ids=[vocab.get(tok,1) for tok in clean_text(text).split()][:MAX_LEN]
    return ids+[0]*(MAX_LEN-len(ids))

X=np.stack([encode(t) for t in df["text"]])
y=df["label"].to_numpy()
print("Vocabulary:",len(vocab)," Sequence shape:",X.shape)
print("Train/Val/Test:",len(train_idx),len(val_idx),len(test_idx))

In [ ]:
class EmailDataset(Dataset):
    def __init__(self,idx):
        self.x=torch.tensor(X[idx],dtype=torch.long)
        self.y=torch.tensor(y[idx],dtype=torch.long)
    def __len__(self): return len(self.y)
    def __getitem__(self,i): return self.x[i],self.y[i]

train_loader=DataLoader(EmailDataset(train_idx),batch_size=64,shuffle=True)
val_loader=DataLoader(EmailDataset(val_idx),batch_size=128)
test_loader=DataLoader(EmailDataset(test_idx),batch_size=128)

## 3. BiLSTM model

In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self,vocab_size,embed_dim=64,hidden_dim=64,num_classes=2):
        super().__init__()
        self.embedding=nn.Embedding(vocab_size,embed_dim,padding_idx=0)
        self.lstm=nn.LSTM(embed_dim,hidden_dim,batch_first=True,bidirectional=True)
        self.dropout=nn.Dropout(.35)
        self.fc=nn.Linear(hidden_dim*2,num_classes)
    def forward(self,x):
        x=self.embedding(x)
        _,(h,_)=self.lstm(x)
        h=torch.cat([h[-2],h[-1]],dim=1)
        return self.fc(self.dropout(h))

model=BiLSTMClassifier(len(vocab)).to(device)
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=1e-3)
print(model)

## 4. Train and validate

In [ ]:
history={"loss":[],"val_loss":[],"accuracy":[],"val_accuracy":[]}
best_state=None; best_val_loss=float("inf"); bad_epochs=0
EPOCHS=12; PATIENCE=3

for epoch in range(EPOCHS):
    model.train(); total_loss=correct=total=0
    for xb,yb in train_loader:
        xb,yb=xb.to(device),yb.to(device)
        optimizer.zero_grad()
        logits=model(xb); loss=criterion(logits,yb)
        loss.backward(); optimizer.step()
        total_loss+=loss.item()*len(yb); correct+=(logits.argmax(1)==yb).sum().item(); total+=len(yb)
    train_loss=total_loss/total; train_acc=correct/total

    model.eval(); val_loss=val_correct=val_total=0
    with torch.no_grad():
        for xb,yb in val_loader:
            xb,yb=xb.to(device),yb.to(device)
            logits=model(xb); loss=criterion(logits,yb)
            val_loss+=loss.item()*len(yb); val_correct+=(logits.argmax(1)==yb).sum().item(); val_total+=len(yb)
    val_loss/=val_total; val_acc=val_correct/val_total
    history["loss"].append(train_loss); history["val_loss"].append(val_loss)
    history["accuracy"].append(train_acc); history["val_accuracy"].append(val_acc)
    print(f"Epoch {epoch+1:02d}: loss={train_loss:.4f} acc={train_acc:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

    if val_loss<best_val_loss:
        best_val_loss=val_loss
        best_state={k:v.detach().cpu().clone() for k,v in model.state_dict().items()}
        bad_epochs=0
    else:
        bad_epochs+=1
    if bad_epochs>=PATIENCE:
        print("Early stopping."); break

model.load_state_dict(best_state)
torch.save({"model_state_dict":model.state_dict(),"vocab":vocab},"ela1_bilstm.pt")

## 5. Loss and accuracy curves

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(12,4))
axes[0].plot(history["loss"],label="Train"); axes[0].plot(history["val_loss"],label="Validation")
axes[0].set_title("Loss"); axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("Loss"); axes[0].legend()
axes[1].plot(history["accuracy"],label="Train"); axes[1].plot(history["val_accuracy"],label="Validation")
axes[1].set_title("Accuracy"); axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("Accuracy"); axes[1].legend()
plt.tight_layout(); plt.show()

## 6. Test metrics and confusion matrix

In [ ]:
model.eval(); y_true=[]; y_pred=[]
with torch.no_grad():
    for xb,yb in test_loader:
        logits=model(xb.to(device))
        y_pred.extend(logits.argmax(1).cpu().numpy()); y_true.extend(yb.numpy())

accuracy=accuracy_score(y_true,y_pred)
precision,recall,f1,_=precision_recall_fscore_support(y_true,y_pred,average="binary",pos_label=1)
print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1-score : {f1:.4f}")
print(classification_report(y_true,y_pred,target_names=["Legitimate","Phishing/Malicious"]))

cm=confusion_matrix(y_true,y_pred)
plt.figure(figsize=(5,4)); plt.imshow(cm,interpolation="nearest")
plt.xticks([0,1],["Legitimate","Phishing"]); plt.yticks([0,1],["Legitimate","Phishing"])
for i in range(2):
    for j in range(2): plt.text(j,i,str(cm[i,j]),ha="center",va="center")
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title("Test Confusion Matrix"); plt.colorbar()
plt.tight_layout(); plt.show()

## 7. Raw email inference

In [ ]:
def predict_email(raw_email):
    model.eval()
    x=torch.tensor([encode(raw_email)],dtype=torch.long).to(device)
    with torch.no_grad(): probs=torch.softmax(model(x),dim=1)[0]
    label="Phishing/Malicious" if probs[1]>=.5 else "Legitimate"
    return label,float(probs[1])

samples=[
"Subject: Security alert. Verify your account immediately at http://secure-login.example.com",
"Subject: Weekly report. Please find the updated report attached. Thank you."
]
for email in samples:
    label,p=predict_email(email)
    print("\nEMAIL:",email,"\nPREDICTION:",label,"\nPHISHING PROBABILITY:",round(p,4))

## 8. Conclusion and limitations

The project demonstrates an end-to-end LSTM text-classification pipeline for raw email payloads.

**Limitation:** the corpus is synthetic and template-driven. The reported metrics should therefore be treated as a demonstration of implementation, not as production phishing-detection performance. A future version should train and test on real email/phishing datasets and evaluate generalisation on an unseen external dataset.